# Camera Image Analysis with GPT-4 Vision

This notebook analyzes CCTV images using GPT-4 Vision API to extract meaningful information about traffic, pedestrians, activities, etc.

In [2]:
import os
import base64
import requests
import json
from pathlib import Path
from PIL import Image
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Azure OpenAI Configuration
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_MODEL = os.getenv("AZURE_OPENAI_MODEL")

# Build Azure API URL
if AZURE_OPENAI_ENDPOINT:
    API_URL = f"{AZURE_OPENAI_ENDPOINT}openai/deployments/{AZURE_OPENAI_MODEL}/chat/completions?api-version={AZURE_OPENAI_API_VERSION}"
else:
    API_URL = None

# Check configuration
if not all([AZURE_OPENAI_API_KEY, AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_VERSION, AZURE_OPENAI_MODEL]):
    print("⚠️ Missing Azure OpenAI configuration in .env file")
    print("Required: AZURE_OPENAI_API_KEY, AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_VERSION, AZURE_OPENAI_MODEL")
else:
    print("✅ Azure OpenAI configuration loaded")
    print(f"🔗 Endpoint: {AZURE_OPENAI_ENDPOINT}")
    print(f"📋 Model: {AZURE_OPENAI_MODEL}")
    print(f"📅 API Version: {AZURE_OPENAI_API_VERSION}")

✅ Azure OpenAI configuration loaded
🔗 Endpoint: https://openaiendpoint-uk-1.openai.azure.com/
📋 Model: gpt-4.1
📅 API Version: 2025-04-01-preview


In [ ]:
default_prompt = """
Analyze this CCTV camera image and provide detailed information about:
1. Number of vehicles (cars, trucks, buses, motorcycles) - count each type
2. Number of pedestrians visible
3. Traffic conditions (light, moderate, heavy, congested)
4. Weather conditions (sunny, cloudy, rainy, night, etc.)
5. Time of day estimate (morning, midday, afternoon, evening, night)
6. Any unusual activities or incidents
7. Overall scene description

Format your response as JSON with these fields:
{
    "vehicles": {"cars": 0, "trucks": 0, "buses": 0, "motorcycles": 0},
    "pedestrians": 0,
    "traffic_level": "light|moderate|heavy|congested",
    "weather": "description",
    "time_of_day": "morning|midday|afternoon|evening|night",
    "incidents": "description or none",
    "scene_description": "brief description"
}
"""

In [ ]:
def encode_image_to_base64(image_path):
    """Convert image to base64 string for API"""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def analyze_camera_image(image_path, prompt=None):
    """
    Analyze camera image using Azure OpenAI GPT-4 Vision API
    
    Args:
        image_path (str): Path to the image file
        prompt (str): Custom prompt for analysis
    
    Returns:
        dict: API response with analysis
    """
    if not AZURE_OPENAI_API_KEY or not API_URL:
        return {"error": "Azure OpenAI configuration missing"}
    
    # Default prompt for CCTV analysis
    if prompt is None:
        prompt = """
        Analyze this CCTV camera image and provide detailed information about:
        1. Number of vehicles (cars, trucks, buses, motorcycles) - count each type
        2. Number of pedestrians visible
        3. Traffic conditions (light, moderate, heavy, congested)
        4. Weather conditions (sunny, cloudy, rainy, night, etc.)
        5. Time of day estimate (morning, midday, afternoon, evening, night)
        6. Any unusual activities or incidents
        7. Overall scene description
        
        Format your response as JSON with these fields:
        {
            "vehicles": {"cars": 0, "trucks": 0, "buses": 0, "motorcycles": 0},
            "pedestrians": 0,
            "traffic_level": "light|moderate|heavy|congested",
            "weather": "description",
            "time_of_day": "morning|midday|afternoon|evening|night",
            "incidents": "description or none",
            "scene_description": "brief description"
        }
        """
    
    # Encode image
    try:
        base64_image = encode_image_to_base64(image_path)
    except Exception as e:
        return {"error": f"Failed to encode image: {str(e)}"}
    
    # Prepare Azure API request headers
    headers = {
        "Content-Type": "application/json",
        "api-key": AZURE_OPENAI_API_KEY  # Azure uses api-key instead of Authorization
    }
    
    payload = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/jpeg;base64,{base64_image}",
                            "detail": "high"
                        }
                    }
                ]
            }
        ],
        "max_tokens": 1000,
        "temperature": 0.1  # Low temperature for consistent analysis
    }
    
    try:
        print("🔄 Calling Azure OpenAI API...")
        response = requests.post(API_URL, headers=headers, json=payload, timeout=60)
        response.raise_for_status()
        
        result = response.json()
        return {
            "success": True,
            "analysis": result["choices"][0]["message"]["content"],
            "usage": result.get("usage", {}),
            "timestamp": datetime.now().isoformat(),
            "image_path": str(image_path),
            "model": AZURE_OPENAI_MODEL
        }
        
    except requests.exceptions.RequestException as e:
        return {"error": f"Azure API request failed: {str(e)}"}
    except Exception as e:
        return {"error": f"Unexpected error: {str(e)}"}

In [ ]:
result = analyze_camera_image("/home/tbenes/multimodal-cctv-surveilance/camera_images/holesovicky/camera_500067_20251117_155024.jpg")

🔄 Calling Azure OpenAI API...


In [5]:
result

{'success': True,
 'analysis': '{\n    "vehicles": {\n        "cars": 24,\n        "trucks": 0,\n        "buses": 0,\n        "motorcycles": 0\n    },\n    "pedestrians": 0,\n    "traffic_level": "moderate",\n    "weather": "cloudy, roads appear slightly wet",\n    "time_of_day": "afternoon",\n    "incidents": "none",\n    "scene_description": "A multi-lane urban road with moderate car traffic, no visible trucks, buses, or motorcycles. No pedestrians are present. The weather is cloudy and the road surface looks damp, possibly after recent rain. Surrounding area includes green spaces and buildings."\n}',
 'usage': {'completion_tokens': 145,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens': 867,
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
  'total_tokens': 1012},
 'timestamp': '2025-11-24T19:58:38.480014',
 'image_path': '/home/tbenes/multimodal-c

In [8]:
bridge_prompt = """
Analyze this CCTV camera image and provide detailed information about:
2. Number of pedestrians visible
3. return an overcrowdedness level with a number from 1 to 10 (1 - empty, 10 - extremely overcrowded)
4. Weather conditions (sunny, cloudy, rainy, night, etc.)
5. Time of day estimate (morning, midday, afternoon, evening, night)
6. Any unusual activities or incidents
7. Overall scene description

Format your response as JSON with these fields:
{
    "pedestrians": 0,
    "overcrowdedness_level": 0,
    "weather": "description",
    "time_of_day": "morning|midday|afternoon|evening|night",
    "incidents": "description or none",
    "scene_description": "brief description"
}
"""

In [9]:
result_bridge = analyze_camera_image("/home/tbenes/multimodal-cctv-surveilance/camera_images/charles_bride/camera_101200_20251117_141239.jpg", bridge_prompt)

🔄 Calling Azure OpenAI API...


In [10]:
result_bridge

{'success': True,
 'analysis': '{\n    "pedestrians": 100,\n    "overcrowdedness_level": 5,\n    "weather": "cloudy",\n    "time_of_day": "afternoon",\n    "incidents": "none",\n    "scene_description": "A historic bridge in Prague with a moderate crowd of pedestrians, cloudy skies, and city buildings in the background. No unusual activities or incidents visible."\n}',
 'usage': {'completion_tokens': 83,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens': 807,
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
  'total_tokens': 890},
 'timestamp': '2025-11-24T20:03:14.647944',
 'image_path': '/home/tbenes/multimodal-cctv-surveilance/camera_images/charles_bride/camera_101200_20251117_141239.jpg',
 'model': 'gpt-4.1'}

In [ ]:
airport_prompt = """
Analyze this CCTV camera image and provide detailed information about what states the airport apron is in:
1. plane is parked at the gate (yes/no)
2. plane has its boarding bridge connected (yes/no)
3. plane has its cargo doors open (yes/no) - for both front and rear cargo doors
4. the ground service guys are hauling luggage or cargo to/from the plane (yes/no)
5. plane refueling status (refueling in progress/not refueling)
6. weather conditions (sunny, cloudy, rainy, night, etc.)
7. time of day estimate (morning, midday, afternoon, evening, night)
8. Any unusual activities or incidents

Format your response as JSON with these fields:
{
    "plane_at_gate": "yes|no",
    "boarding_bridge_connected": "yes|no",
    "cargo_doors_open": {"front": "yes|no", "rear": "yes|no"},
    "ground_service_active": "yes|no",
    "refueling_status": "refueling in progress|not refueling",
    "weather": "description",
    "time_of_day": "morning|midday|afternoon|evening|night",
    "incidents": "description or none",
    "scene_description": "brief description"
}
"""

In [15]:
result_airport = analyze_camera_image("/home/tbenes/multimodal-cctv-surveilance/camera_images/airport/vlcsnap-2025-11-24-20h16m27s521.png", airport_prompt)

🔄 Calling Azure OpenAI API...


In [16]:
result_airport

{'success': True,
 'analysis': '{\n    "plane_at_gate": "yes",\n    "boarding_bridge_connected": "yes",\n    "cargo_doors_open": {"front": "yes", "rear": "no"},\n    "ground_service_active": "yes",\n    "refueling_status": "not refueling",\n    "weather": "clear and sunny",\n    "time_of_day": "morning",\n    "incidents": "none",\n    "scene_description": "A Japan Airlines aircraft is parked at the gate with the boarding bridge connected. Ground service vehicles and personnel are present, with luggage or cargo being handled at the front cargo door. The weather is clear and sunny, and the apron is calm with no unusual activities."\n}',
 'usage': {'completion_tokens': 145,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens': 1185,
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
  'total_tokens': 1330},
 'timestamp': '2025-11-24T20:20:58.970224',
 'image_

In [17]:
result_airport2 = analyze_camera_image("/home/tbenes/multimodal-cctv-surveilance/camera_images/airport/vlcsnap-2025-11-24-20h22m00s182.png", airport_prompt)
result_airport2

🔄 Calling Azure OpenAI API...


{'success': True,
 'analysis': '{\n    "plane_at_gate": "yes",\n    "boarding_bridge_connected": "yes",\n    "cargo_doors_open": {"front": "no", "rear": "no"},\n    "ground_service_active": "yes",\n    "refueling_status": "not refueling",\n    "weather": "clear and sunny",\n    "time_of_day": "morning",\n    "incidents": "none",\n    "scene_description": "A Japan Airlines aircraft is parked at the gate with the boarding bridge connected. Ground service vehicles and personnel are present, but no cargo doors are visibly open and no refueling is taking place. The weather is clear and sunny, with shadows indicating morning time."\n}',
 'usage': {'completion_tokens': 144,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens': 1185,
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
  'total_tokens': 1329},
 'timestamp': '2025-11-24T20:22:57.745528',
 'image_path

In [18]:
result_airport3 = analyze_camera_image("/home/tbenes/multimodal-cctv-surveilance/camera_images/airport/vlcsnap-2025-11-24-20h24m42s638.png", airport_prompt)
result_airport3

🔄 Calling Azure OpenAI API...


{'success': True,
 'analysis': '{\n    "plane_at_gate": "no",\n    "boarding_bridge_connected": "no",\n    "cargo_doors_open": {"front": "no", "rear": "no"},\n    "ground_service_active": "no",\n    "refueling_status": "not refueling",\n    "weather": "clear and sunny",\n    "time_of_day": "morning",\n    "incidents": "none",\n    "scene_description": "A Japan Airlines plane is parked on the apron away from the gate, with no boarding bridge connected and no visible ground service activity. The weather is clear and sunny, and the apron is quiet with a few personnel standing nearby."\n}',
 'usage': {'completion_tokens': 138,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens': 1185,
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0},
  'total_tokens': 1323},
 'timestamp': '2025-11-24T20:25:35.764131',
 'image_path': '/home/tbenes/multimodal-cctv-surveilance/

## Usage Instructions

1. **Environment is already configured** via `.env` file:
   - ✅ Azure OpenAI API Key
   - ✅ Azure endpoint  
   - ✅ API version and model

2. **Install required packages**:
   ```bash
   pip install python-dotenv
   ```

3. **Test single image**:
   ```python
   test_result = test_single_image()
   ```

4. **Analyze multiple images**:
   ```python
   # Analyze Charles Bridge images
   results = batch_analyze_images("../../camera_images/charles_bridge", max_images=5)
   
   # Analyze all locations
   locations = ["charles_bridge", "holesovicky", "budejovicka"]
   for location in locations:
       results = batch_analyze_images(f"../../camera_images/{location}", max_images=3)
   ```

5. **Custom analysis**:
   ```python
   custom_prompt = "Count only the red cars in this image and describe the weather."
   result = analyze_camera_image("path/to/image.jpg", prompt=custom_prompt)
   ```

**Azure Configuration Details:**
- **Endpoint**: `https://openaiendpoint-uk-1.openai.azure.com/`
- **Model**: `gpt-4.1` 
- **API Version**: `2025-04-01-preview`

The analysis will return structured data about traffic, pedestrians, weather, and incidents detected in CCTV images.